Uma análise feita sobre as rotas atuais demonstram que atualmente o formato de distribuição causará atrasos na maior parte das entregas. Esta análise pode ser vista pelo link: [1. Analysis - Rotas](https://github.com/gxlobato/DunderMifflin/blob/main/notebooks/1.%20Analysis%20-%20Rotas.ipynb)

Neste notebook pretendo explorar 3 cenários para a reestruturação

###Cenário A — Manter 4 armazéns, só mudar a regra (Centro-Oeste → Sudeste + Sul)
Reaproveita o que existe, mas talvez não resolva os piores casos (Centro-Oeste é geograficamente vasto).

###Cenário B — Realocar um armazém existente pra mais perto do Centro-Oeste
Por exemplo, trocar Recife (Nordeste) por Goiânia ou Brasília. Isso resolve o Centro-Oeste, mas potencialmente piora o Nordeste (que ficaria sem armazém próprio, teria que ser coberto por outro).

###Cenário C — Adicionar um 5º armazém no Centro-Oeste
Resolve o Centro-Oeste sem sacrificar nenhuma região existente, mas é mais armazém pra manter (custo/complexidade maior).

### Monta os candidatos e calcula a matriz completa

In [0]:
# %pip install faker
# dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Repos/Users/gabrielalbt@gmail.com/dunder-mifflin")  # ajusta pro caminho real do repo no Databricks
import os

import pandas as pd
from src.shared.municipios import buscar_lat_long

# Forçar reload do módulo para pegar as alterações
import importlib
import src.logistica.distancias
importlib.reload(src.logistica.distancias)
from src.logistica.distancias import calcula_distancias

DIR_RAW = "/Volumes/workspace/dundermiffin/raw"
DIR_SOURCE = "/Volumes/workspace/dundermiffin/source"
DIR_CACHE = "/Volumes/workspace/dundermiffin/cache"

api_key = dbutils.widgets.get('api_key')

def carregar_parquet(caminho):
    """Carrega um DataFrame salvo em Parquet, ou None se o arquivo não existir."""
    if not os.path.exists(caminho):
        return None
    return pd.read_parquet(caminho)

def salvar_parquet(df, caminho):
    """Salva um DataFrame em formato Parquet."""
    os.makedirs(os.path.dirname(caminho), exist_ok=True)
    df.to_parquet(caminho, index=False)

# carrega tudo que já foi gerado pelo pipeline principal, sem recalcular nada
df_municipios = carregar_parquet(f"{DIR_RAW}/municipios.parquet")
df_lat_long = carregar_parquet(f"{DIR_RAW}/lat_long.parquet")
df_armazem = carregar_parquet(f"{DIR_SOURCE}/armazens.parquet")
df_clientes = carregar_parquet(f"{DIR_SOURCE}/clientes.parquet")

for nome, df in [
    ('municipios', df_municipios), ('lat_long', df_lat_long),
    ('armazem', df_armazem), ('clientes', df_clientes)
]:
    if df is None:
        raise FileNotFoundError(f"{nome}.parquet não encontrado — rode o notebook principal primeiro.")

### Monta os candidatos e calcula a matriz completa

In [0]:
# Candidatos a armazém no Centro-Oeste, para testar realocação (Cenário B)
# ou adição de um 5º armazém (Cenário C)
candidatos_extra = [
    {'cidade': 'Goiânia', 'codigo': 'A5'},
    {'cidade': 'Brasília', 'codigo': 'A6'},
]

def montar_candidatos(df_municipios, df_lat_long, candidatos, id_inicial):
    """Monta armazéns candidatos extras, no mesmo formato de montar_armazens()."""
    linhas = []
    for i, c in enumerate(candidatos):
        linha = df_municipios[df_municipios['nome_cidade'] == c['cidade']].head(1)
        if linha.empty:
            raise ValueError(f"Cidade não encontrada: {c['cidade']}")

        codigo_ibge = linha['codigo_ibge'].values[0]
        uf = linha['uf'].values[0]
        regiao = linha['regiao'].values[0]
        latitude, longitude = buscar_lat_long(df_lat_long, codigo_ibge)

        linhas.append({
            'id': id_inicial + i,
            'codigo': c['codigo'],
            'cidade': c['cidade'],
            'uf': uf,
            'regiao': regiao,
            'latitude': latitude,
            'longitude': longitude,
        })
    return pd.DataFrame(linhas)

# junta os 4 armazéns reais + os candidatos, com ids sequenciais (5, 6...)
df_candidatos = montar_candidatos(df_municipios, df_lat_long, candidatos_extra, id_inicial=df_armazem['id'].max() + 1)
df_armazem_todos = pd.concat([df_armazem, df_candidatos], ignore_index=True)

print(df_armazem_todos[['id', 'codigo', 'cidade', 'regiao']])

### Calcula a distância de todo cliente até todo armazém/candidato, em uma única chamada

In [0]:
import requests

# reaproveita calcula_distancias, mas com df_armazem_todos (4 atuais + candidatos)
df_distancias_todos = calcula_distancias(df_armazem_todos, df_clientes, api_key)

# opcional: salvar em cache, já que envolve chamada de API
salvar_parquet(df_distancias_todos, f"{DIR_CACHE}/distancias_cenarios_centro_oeste.parquet")

### Define os 3 cenários como mapas de "quais armazéns cada região pode usar"

In [0]:
# id 1=SP, 2=Curitiba, 3=Recife, 4=Belém, 5=Goiânia, 6=Brasília
cenarios = {
    'A - Corrige regra (CO -> Sudeste+Sul)': {
        'Norte': [4, 3],
        'Nordeste': [3, 4],
        'Centro-Oeste': [1, 2],
        'Sudeste': [1, 2],
        'Sul': [2, 1],
    },
    'B - Realoca Recife -> Goiânia': {
        'Norte': [4],
        'Nordeste': [4, 5],   # Nordeste passa a ser coberto por Belém/Goiânia
        'Centro-Oeste': [5, 1],
        'Sudeste': [1, 2],
        'Sul': [2, 1],
    },
    'C - Mantém 4 + adiciona Goiânia': {
        'Norte': [4, 3],
        'Nordeste': [3, 4],
        'Centro-Oeste': [5, 1],
        'Sudeste': [1, 2],
        'Sul': [2, 1],
    },
}

### Simula cada cenário: para cada cliente, pega a menor distância entre os armazéns permitidos

In [0]:
def simular_cenario(nome, mapa_regioes, df_clientes, df_distancias_todos):
    registros = []
    for _, cliente in df_clientes.iterrows():
        armazens_validos = mapa_regioes.get(cliente['regiao'], [])
        candidatas = df_distancias_todos[
            (df_distancias_todos['id_cliente'] == cliente['id']) &
            (df_distancias_todos['id_armazem'].isin(armazens_validos))
        ]
        if candidatas.empty:
            continue

        melhor = candidatas.sort_values('distancia_metros').iloc[0]
        registros.append({
            'cenario': nome,
            'id_cliente': cliente['id'],
            'regiao': cliente['regiao'],
            'id_armazem_escolhido': melhor['id_armazem'],
            'distancia_km': melhor['distancia_metros'] / 1000,
        })
    return pd.DataFrame(registros)

resultados = []
for nome, mapa in cenarios.items():
    resultados.append(simular_cenario(nome, mapa, df_clientes, df_distancias_todos))

df_comparacao = pd.concat(resultados, ignore_index=True)

### Compara os cenários

In [0]:
resumo = df_comparacao.groupby('cenario').agg(
    media_distancia_km=('distancia_km', 'mean'),
    maior_distancia_km=('distancia_km', 'max'),
    clientes_acima_800km=('distancia_km', lambda x: (x > 800).sum()),
    total_clientes=('distancia_km', 'count'),
).reset_index()

display(resumo)

In [0]:
df_comparacao[df_comparacao['distancia_km'] > 4000]

# Exploração: cobertura de armazém para o Centro-Oeste

A análise de rotas revelou entregas com duração fisicamente inviável, causadas por clientes do Centro-Oeste sendo atendidos por Recife (Nordeste), a milhares de km de distância.

## Cenários testados

| Cenário | Descrição | Distância média (km) | Clientes acima de 800 km |
|---|---|---|---|
| A | Centro-Oeste passa a ser atendido por Sudeste + Sul | 817,7 | 17 |
| B | Realoca Recife → Goiânia | 999,2 | 21 |
| **C** | **Mantém os 4 armazéns e adiciona Goiânia** | **719,4** | **12** |

## Decisão

Seguimos com o **Cenário C**: menor distância média e menos clientes em situação crítica, sem piorar nenhuma região já atendida.

## Pendência encontrada

O cliente de id 14 (região Norte, via Belém) ficou a 4.122 km em todos os cenários — a região Norte tem um problema de cobertura semelhante, ainda não resolvido. Fica para investigação futura.